# PART-1 감정 분석: GPT-2 Fine-tuning on SST & CFIMDB

이 노트북은 GPT-2를 두 가지 감정 분석 데이터셋에서 fine-tuning하고 평가하기 위한 실행 기록입니다.

**포함 내용:**
- GPT-2 아키텍처 구현 검증 (`sanity_check.py` 로직)
- AdamW Optimizer 구현 검증 (`optimizer_test.py` 로직)
- SST(Stanford Sentiment Treebank) 데이터셋: 5-class 감정 분류
- CFIMDB(Compact Fine-grained IMDB) 데이터셋: 2-class 감정 분류
- `last-linear-layer` vs `full-model` fine-tuning 방식 비교
- Dev split 성능 지표, Confusion Matrix, 시각화
- HTML → PDF 내보내기

**기준 모델 (baseline) Dev 정확도:**
| 데이터셋 | Fine-tune 방식 | 기준 정확도 |
|---------|------------|----------|
| SST | last-linear-layer | 0.462 |
| SST | full-model | 0.513 |
| CFIMDB | last-linear-layer | 0.861 |
| CFIMDB | full-model | 0.976 |

## 1. 프로젝트 루트 준비

노트북을 프로젝트 루트 또는 `notebooks/` 안에서 실행해도 동일하게 동작하도록 작업 디렉터리와 import path를 맞춥니다.

In [ ]:
from pathlib import Path
import os
import sys
import csv
import json
from collections import Counter, defaultdict
from datetime import datetime

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'classifier.py').exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / 'classifier.py').exists():
        PROJECT_ROOT = candidate

assert (PROJECT_ROOT / 'classifier.py').exists(), \
    '프로젝트 루트 또는 notebooks/ 폴더에서 실행하세요.'
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

## 2. 소스 코드 불러오기

`classifier.py`, `optimizer.py`, `models/gpt2.py`, `modules/attention.py`, `modules/gpt2_layer.py` 등 PART-I 관련 모든 모듈을 import합니다.

In [ ]:
import importlib
import random
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    precision_recall_fscore_support,
)

# PART-I 핵심 모듈
import classifier
import optimizer as optimizer_module
from optimizer import AdamW
from models.gpt2 import GPT2Model
from modules.attention import CausalSelfAttention
from modules.gpt2_layer import GPT2Layer
import classifier
importlib.reload(classifier)

print(f'torch: {torch.__version__}')
print(f'cuda available: {torch.cuda.is_available()}')
print(f'visible gpus: {torch.cuda.device_count()}')

## 3. GPT-2 구현 검증

구현한 `GPT2Model`의 출력이 HuggingFace 공식 GPT-2 모델과 일치하는지 확인합니다.
- `sanity_check.py`의 `test_gpt2()` 로직을 인라인으로 실행합니다.
- atol=0.1, rtol=0.01 기준으로 비교합니다.

In [ ]:
from transformers import GPT2Model as OpenAIGPT2Model
from utils import model_size_to_params

def verify_gpt2_implementation(model_size='gpt2'):
    sent_ids = torch.tensor([[101, 7592, 2088, 102, 0, 0, 0, 0],
                              [101, 7592, 15756, 2897, 2005, 17953, 2361, 102]])
    att_mask = torch.tensor([[1, 1, 1, 1, 0, 0, 0, 0],
                              [1, 1, 1, 1, 1, 1, 1, 1]])

    openai_model = OpenAIGPT2Model.from_pretrained(model_size)
    openai_model.eval()

    gpt = GPT2Model.from_pretrained(model=model_size, **model_size_to_params(model_size))
    gpt.eval()

    with torch.no_grad():
        our_outputs = gpt(sent_ids, att_mask)
        openai_outputs = openai_model(
            input_ids=sent_ids,
            attention_mask=att_mask,
            output_hidden_states=True,
        ).hidden_states[-1]

    mask = att_mask.unsqueeze(-1)
    our_last = our_outputs['last_hidden_state'] * mask
    openai_last = openai_outputs * mask

    max_diff = (our_last - openai_last).abs().max().item()
    passed = torch.allclose(our_last, openai_last, atol=1e-1, rtol=1e-2)

    return {
        'model_size': model_size,
        'output_shape': list(our_last.shape),
        'max_abs_diff': round(max_diff, 6),
        'passed': passed,
    }

gpt2_check_result = verify_gpt2_implementation('gpt2')
print(json.dumps(gpt2_check_result, indent=2))

if gpt2_check_result['passed']:
    print('\n[PASS] GPT-2 구현이 HuggingFace 공식 모델과 일치합니다.')
else:
    print('\n[FAIL] GPT-2 구현에 문제가 있습니다. modules/attention.py, modules/gpt2_layer.py를 확인하세요.')

## 4. AdamW Optimizer 검증

구현한 `AdamW.step()` 함수가 참조값(`optimizer_test.npy`)과 일치하는지 확인합니다.
- 1000 step 학습 후 모델 가중치를 비교합니다.

In [ ]:
OPTIMIZER_TEST_SEED = 0

def test_adamw_optimizer():
    rng = np.random.default_rng(OPTIMIZER_TEST_SEED)
    torch.manual_seed(OPTIMIZER_TEST_SEED)

    model = torch.nn.Linear(3, 2, bias=False)
    opt = AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
        correct_bias=True,
    )

    for _ in range(1000):
        opt.zero_grad()
        x = torch.FloatTensor(rng.uniform(size=[model.in_features]))
        y_hat = model(x)
        y = torch.Tensor([x[0] + x[1], -x[2]])
        loss = ((y - y_hat) ** 2).sum()
        loss.backward()
        opt.step()

    return model.weight.detach()

ref_path = PROJECT_ROOT / 'optimizer_test.npy'
ref = torch.tensor(np.load(str(ref_path)))
actual = test_adamw_optimizer()

max_diff = (ref - actual).abs().max().item()
passed = torch.allclose(ref, actual, atol=1e-6, rtol=1e-4)

optimizer_result = {
    'reference_weight': ref.tolist(),
    'actual_weight': actual.tolist(),
    'max_abs_diff': round(max_diff, 10),
    'passed': passed,
}
print(json.dumps(optimizer_result, indent=2))

if passed:
    print('\n[PASS] AdamW Optimizer 구현이 참조값과 일치합니다.')
else:
    print('\n[FAIL] Optimizer step() 구현을 확인하세요.')

## 5. 실행 설정

SST와 CFIMDB 각각에 대해 `last-linear-layer`와 `full-model` 두 가지 fine-tuning 방식을 설정합니다.
- `RUN_*_TRAIN = True`로 바꾸면 해당 모델을 새로 학습합니다.
- 기존 checkpoint가 있으면 학습을 건너뛰고 평가만 수행합니다.

In [ ]:
# ─── 학습 여부 ─────────────────────────────────────────────
RUN_SST_LAST_LINEAR_TRAIN  = False
RUN_SST_FULL_MODEL_TRAIN   = False
RUN_CFIMDB_LAST_LINEAR_TRAIN = False
RUN_CFIMDB_FULL_MODEL_TRAIN  = False

ALLOW_CPU_EXECUTION = False

# ─── 공통 하이퍼파라미터 ──────────────────────────────────────
DEVICE_NAME = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 11711
SST_EPOCHS = 10
CFIMDB_EPOCHS = 10
SST_BATCH_SIZE = 64
CFIMDB_BATCH_SIZE = 8
HIDDEN_DROPOUT = 0.3

# ─── checkpoint / prediction 경로 ─────────────────────────
SST_LAST_LINEAR_CKPT   = 'sst-last-linear-classifier.pt'
SST_FULL_MODEL_CKPT    = 'sst-classifier.pt'
CFIMDB_LAST_LINEAR_CKPT = 'cfimdb-last-linear-classifier.pt'
CFIMDB_FULL_MODEL_CKPT  = 'cfimdb-classifier.pt'

configs = {
    'sst_last_linear': SimpleNamespace(
        label='SST / last-linear-layer',
        dataset='sst',
        fine_tune_mode='last-linear-layer',
        filepath=SST_LAST_LINEAR_CKPT,
        train='data/ids-sst-train.csv',
        dev='data/ids-sst-dev.csv',
        test='data/ids-sst-test-student.csv',
        dev_out='predictions/last-linear-layer-sst-dev-out.csv',
        test_out='predictions/last-linear-layer-sst-test-out.csv',
        lr=1e-3,
        epochs=SST_EPOCHS,
        batch_size=SST_BATCH_SIZE,
        hidden_dropout_prob=HIDDEN_DROPOUT,
        use_gpu=(DEVICE_NAME == 'cuda'),
        run_train=RUN_SST_LAST_LINEAR_TRAIN,
    ),
    'sst_full_model': SimpleNamespace(
        label='SST / full-model',
        dataset='sst',
        fine_tune_mode='full-model',
        filepath=SST_FULL_MODEL_CKPT,
        train='data/ids-sst-train.csv',
        dev='data/ids-sst-dev.csv',
        test='data/ids-sst-test-student.csv',
        dev_out='predictions/full-model-sst-dev-out.csv',
        test_out='predictions/full-model-sst-test-out.csv',
        lr=1e-5,
        epochs=SST_EPOCHS,
        batch_size=SST_BATCH_SIZE,
        hidden_dropout_prob=HIDDEN_DROPOUT,
        use_gpu=(DEVICE_NAME == 'cuda'),
        run_train=RUN_SST_FULL_MODEL_TRAIN,
    ),
    'cfimdb_last_linear': SimpleNamespace(
        label='CFIMDB / last-linear-layer',
        dataset='cfimdb',
        fine_tune_mode='last-linear-layer',
        filepath=CFIMDB_LAST_LINEAR_CKPT,
        train='data/ids-cfimdb-train.csv',
        dev='data/ids-cfimdb-dev.csv',
        test='data/ids-cfimdb-test-student.csv',
        dev_out='predictions/last-linear-layer-cfimdb-dev-out.csv',
        test_out='predictions/last-linear-layer-cfimdb-test-out.csv',
        lr=1e-3,
        epochs=CFIMDB_EPOCHS,
        batch_size=CFIMDB_BATCH_SIZE,
        hidden_dropout_prob=HIDDEN_DROPOUT,
        use_gpu=(DEVICE_NAME == 'cuda'),
        run_train=RUN_CFIMDB_LAST_LINEAR_TRAIN,
    ),
    'cfimdb_full_model': SimpleNamespace(
        label='CFIMDB / full-model',
        dataset='cfimdb',
        fine_tune_mode='full-model',
        filepath=CFIMDB_FULL_MODEL_CKPT,
        train='data/ids-cfimdb-train.csv',
        dev='data/ids-cfimdb-dev.csv',
        test='data/ids-cfimdb-test-student.csv',
        dev_out='predictions/full-model-cfimdb-dev-out.csv',
        test_out='predictions/full-model-cfimdb-test-out.csv',
        lr=1e-5,
        epochs=CFIMDB_EPOCHS,
        batch_size=CFIMDB_BATCH_SIZE,
        hidden_dropout_prob=HIDDEN_DROPOUT,
        use_gpu=(DEVICE_NAME == 'cuda'),
        run_train=RUN_CFIMDB_FULL_MODEL_TRAIN,
    ),
}

run_config_summary = {
    key: {
        'label': cfg.label,
        'fine_tune_mode': cfg.fine_tune_mode,
        'checkpoint': cfg.filepath,
        'run_train': cfg.run_train,
        'lr': cfg.lr,
        'epochs': cfg.epochs,
        'batch_size': cfg.batch_size,
    }
    for key, cfg in configs.items()
}
print(json.dumps(run_config_summary, indent=2, ensure_ascii=False))

## 6. 데이터와 산출물 확인

실행 전 train/dev/test 데이터, checkpoint, prediction 파일 상태를 확인합니다.

In [ ]:
def file_status(path):
    p = PROJECT_ROOT / path
    if not p.exists():
        return {'path': path, 'exists': False}
    return {
        'path': path,
        'exists': True,
        'size_mb': round(p.stat().st_size / (1024 * 1024), 3),
        'modified': datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec='seconds'),
    }

artifact_status = {
    'data': {
        'sst_train': file_status('data/ids-sst-train.csv'),
        'sst_dev':   file_status('data/ids-sst-dev.csv'),
        'sst_test':  file_status('data/ids-sst-test-student.csv'),
        'cfimdb_train': file_status('data/ids-cfimdb-train.csv'),
        'cfimdb_dev':   file_status('data/ids-cfimdb-dev.csv'),
        'cfimdb_test':  file_status('data/ids-cfimdb-test-student.csv'),
    },
    'checkpoints': {
        key: file_status(cfg.filepath) for key, cfg in configs.items()
    },
    'predictions': {},
}
for key, cfg in configs.items():
    artifact_status['predictions'][key] = {
        'dev': file_status(cfg.dev_out),
        'test': file_status(cfg.test_out),
    }

print(json.dumps(artifact_status, indent=2, ensure_ascii=False))

## 7. 데이터 로딩 기록

SST와 CFIMDB 데이터셋의 split 크기와 label distribution을 확인합니다.

In [ ]:
from classifier import load_data

# SST
sst_train_data, sst_num_labels = load_data('data/ids-sst-train.csv', 'train')
sst_dev_data = load_data('data/ids-sst-dev.csv', 'valid')
sst_test_data = load_data('data/ids-sst-test-student.csv', 'test')

# CFIMDB
cfimdb_train_data, cfimdb_num_labels = load_data('data/ids-cfimdb-train.csv', 'train')
cfimdb_dev_data = load_data('data/ids-cfimdb-dev.csv', 'valid')
cfimdb_test_data = load_data('data/ids-cfimdb-test-student.csv', 'test')

SST_LABEL_NAMES = {
    0: 'Very Negative',
    1: 'Negative',
    2: 'Neutral',
    3: 'Positive',
    4: 'Very Positive',
}
CFIMDB_LABEL_NAMES = {0: 'Negative', 1: 'Positive'}

data_summary = {
    'SST': {
        'num_labels': sst_num_labels,
        'train': len(sst_train_data),
        'dev': len(sst_dev_data),
        'test': len(sst_test_data),
        'train_label_counts': dict(sorted(Counter([r[1] for r in sst_train_data]).items())),
        'dev_label_counts': dict(sorted(Counter([r[1] for r in sst_dev_data]).items())),
    },
    'CFIMDB': {
        'num_labels': cfimdb_num_labels,
        'train': len(cfimdb_train_data),
        'dev': len(cfimdb_dev_data),
        'test': len(cfimdb_test_data),
        'train_label_counts': dict(sorted(Counter([r[1] for r in cfimdb_train_data]).items())),
        'dev_label_counts': dict(sorted(Counter([r[1] for r in cfimdb_dev_data]).items())),
    },
}
print(json.dumps(data_summary, indent=2, ensure_ascii=False))

In [ ]:
# 데이터 분포 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Dataset Label Distribution', fontsize=14, fontweight='bold')

SST_COLORS = ['#ef4444', '#f97316', '#6b7280', '#22c55e', '#16a34a']
CFIMDB_COLORS = ['#ef4444', '#22c55e']

def plot_label_dist(ax, label_counts, label_names, colors, title):
    labels = sorted(label_counts.keys())
    counts = [label_counts[l] for l in labels]
    names = [label_names.get(l, str(l)) for l in labels]
    bars = ax.bar(names, counts, color=colors[:len(labels)])
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.25)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts)*0.01,
                f'{count:,}', ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, max(counts) * 1.12)

plot_label_dist(axes[0][0], data_summary['SST']['train_label_counts'],
                SST_LABEL_NAMES, SST_COLORS, 'SST Train Label Distribution')
plot_label_dist(axes[0][1], data_summary['SST']['dev_label_counts'],
                SST_LABEL_NAMES, SST_COLORS, 'SST Dev Label Distribution')
plot_label_dist(axes[1][0], data_summary['CFIMDB']['train_label_counts'],
                CFIMDB_LABEL_NAMES, CFIMDB_COLORS, 'CFIMDB Train Label Distribution')
plot_label_dist(axes[1][1], data_summary['CFIMDB']['dev_label_counts'],
                CFIMDB_LABEL_NAMES, CFIMDB_COLORS, 'CFIMDB Dev Label Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# 문장 길이 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Sentence Length Distribution (Train Split)', fontsize=13, fontweight='bold')

def plot_length_hist(ax, data, title, color):
    lengths = [len(row[0].split()) for row in data]
    ax.hist(lengths, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Word Count')
    ax.set_ylabel('Frequency')
    ax.axvline(np.mean(lengths), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {np.mean(lengths):.1f}')
    ax.legend()
    ax.grid(axis='y', alpha=0.25)

plot_length_hist(axes[0], sst_train_data, 'SST Train', '#2563eb')
plot_length_hist(axes[1], cfimdb_train_data, 'CFIMDB Train', '#16a34a')

plt.tight_layout()
plt.show()

## 8. 학습 및 테스트 실행

아래 셀 하나로 4가지 구성(SST/CFIMDB × last-linear/full-model)을 순차 실행합니다.
- `RUN_*_TRAIN = True`로 설정된 경우만 학습을 수행합니다.
- 기존 prediction 파일이 있으면 평가 셀에서 바로 사용합니다.

In [ ]:
def require_device(will_run):
    if will_run and not torch.cuda.is_available() and not ALLOW_CPU_EXECUTION:
        raise RuntimeError('GPU가 없습니다. CPU 실행을 허용하려면 ALLOW_CPU_EXECUTION = True로 바꾸세요.')

def require_checkpoint(cfg):
    if not cfg.run_train and not (PROJECT_ROOT / cfg.filepath).exists():
        raise FileNotFoundError(f'checkpoint 없음: {cfg.filepath}')


def run_sentiment_pipeline(cfg):
    print(f'\n===== {cfg.label} =====')
    require_checkpoint(cfg)

    if cfg.run_train:
        print(f'Training: {cfg.filepath}')
        classifier.seed_everything(SEED)
        classifier.train(cfg)
    else:
        print('Skipping training (skip_train)')

    if (PROJECT_ROOT / cfg.filepath).exists():
        print(f'Testing: {cfg.filepath}')
        classifier.test(cfg)
    else:
        print(f'Checkpoint not found, skipping test: {cfg.filepath}')


will_run_any = any(cfg.run_train for cfg in configs.values())
require_device(will_run_any)

for key, cfg in configs.items():
    run_sentiment_pipeline(cfg)

print('\n===== 완료 =====')

## 9. Prediction 파일 요약

실행 후 dev/test prediction 파일의 row 수와 label distribution을 확인합니다.

In [ ]:
def summarize_prediction_file(path, preview_rows=3):
    p = PROJECT_ROOT / path
    if not p.exists():
        return {'path': path, 'exists': False}

    counts = Counter()
    preview = []
    rows = 0
    with p.open(newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows += 1
            counts[int(row.get('Predicted_Sentiment', -1))] += 1
            if len(preview) < preview_rows:
                preview.append(row)
    return {
        'path': path,
        'exists': True,
        'rows': rows,
        'label_counts': dict(sorted(counts.items())),
        'preview': preview,
    }

prediction_summary = {}
for key, cfg in configs.items():
    prediction_summary[key] = {
        'dev': summarize_prediction_file(cfg.dev_out),
        'test': summarize_prediction_file(cfg.test_out),
    }

print(json.dumps(prediction_summary, indent=2, ensure_ascii=False))

## 10. 성능 평가

label이 있는 dev split에서 accuracy, macro-F1, label별 precision/recall/F1, confusion matrix를 계산합니다.

In [ ]:
def load_dev_records(dev_path):
    records = {}
    with open(PROJECT_ROOT / dev_path, newline='') as f:
        for row in csv.DictReader(f, delimiter='\t'):
            sent_id = row['id'].lower().strip()
            label = int(row['sentiment'].strip())
            records[sent_id] = {'id': sent_id, 'sentence': row['sentence'], 'label': label}
    return records

def load_predictions(pred_path):
    preds = {}
    p = PROJECT_ROOT / pred_path
    if not p.exists():
        return preds
    with p.open(newline='') as f:
        for row in csv.DictReader(f):
            preds[row['id'].lower().strip()] = int(row['Predicted_Sentiment'])
    return preds

def evaluate_model(label, dev_path, pred_path):
    dev_records = load_dev_records(dev_path)
    predictions = load_predictions(pred_path)

    if not predictions:
        return None

    matched = [sid for sid in dev_records if sid in predictions]
    if not matched:
        return None

    y_true = [dev_records[sid]['label'] for sid in matched]
    y_pred = [predictions[sid] for sid in matched]
    all_labels = sorted(set(y_true))

    prec, rec, f1_per, support = precision_recall_fscore_support(
        y_true, y_pred, labels=all_labels, zero_division=0)
    matrix = confusion_matrix(y_true, y_pred, labels=all_labels).tolist()

    return {
        'label': label,
        'dev_examples': len(dev_records),
        'matched': len(matched),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'per_label_f1': {str(l): round(f, 4) for l, f in zip(all_labels, f1_per)},
        'confusion_matrix': matrix,
        'all_labels': all_labels,
        'y_true': y_true,
        'y_pred': y_pred,
    }

evaluation_results = {}
for key, cfg in configs.items():
    result = evaluate_model(cfg.label, cfg.dev, cfg.dev_out)
    if result:
        evaluation_results[key] = result
        print(f'{cfg.label}: acc={result["accuracy"]:.4f}, macro-F1={result["macro_f1"]:.4f}')
    else:
        print(f'{cfg.label}: prediction file not found, skipping.')

# Comparison table
rows = []
for key, res in evaluation_results.items():
    rows.append({
        'Model': res['label'],
        'Accuracy': round(res['accuracy'], 4),
        'Macro-F1': round(res['macro_f1'], 4),
        'Examples': res['matched'],
    })
if rows:
    display(pd.DataFrame(rows).set_index('Model'))

## 11. 시각화

성능 지표, confusion matrix, label별 F1을 한 번에 비교합니다.

In [ ]:
if not evaluation_results:
    print('평가 결과가 없습니다. prediction 파일을 먼저 생성하세요.')
else:
    # ─── 1. Accuracy & Macro-F1 비교 ─────────────────────────
    result_list = list(evaluation_results.values())
    short_labels = [r['label'].replace(' / ', '\n') for r in result_list]
    accuracies = [r['accuracy'] for r in result_list]
    macro_f1s = [r['macro_f1'] for r in result_list]

    x = np.arange(len(result_list))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('PART-1 Sentiment Analysis: Dev Performance', fontsize=14, fontweight='bold')

    bars_acc = axes[0].bar(x - width/2, accuracies, width, label='Accuracy', color='#2563eb')
    bars_f1  = axes[0].bar(x + width/2, macro_f1s,  width, label='Macro-F1', color='#16a34a')
    axes[0].set_title('Accuracy vs Macro-F1')
    axes[0].set_ylabel('Score')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(short_labels, fontsize=9)
    axes[0].set_ylim(0, 1.15)
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.25)
    for bar in list(bars_acc) + list(bars_f1):
        axes[0].bar_label(
            axes[0].containers[list(axes[0].containers).index(bar.get_container() if hasattr(bar, 'get_container') else bar)],
            fmt='%.3f', padding=3, fontsize=8
        ) if False else None
    for bar, val in zip(bars_acc, accuracies):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    for bar, val in zip(bars_f1, macro_f1s):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    # ─── 2. SST vs CFIMDB accuracy 비교 ─────────────────────
    sst_results   = [(k, v) for k, v in evaluation_results.items() if 'sst' in k]
    cfimdb_results = [(k, v) for k, v in evaluation_results.items() if 'cfimdb' in k]

    group_labels = []
    sst_accs, cfimdb_accs = [], []
    modes = ['last-linear-layer', 'full-model']
    mode_labels = ['Last Linear Layer', 'Full Model']

    sst_acc_map = {v['label'].split(' / ')[1]: v['accuracy'] for _, v in sst_results}
    cf_acc_map  = {v['label'].split(' / ')[1]: v['accuracy'] for _, v in cfimdb_results}

    for mode in modes:
        sst_accs.append(sst_acc_map.get(mode, 0))
        cfimdb_accs.append(cf_acc_map.get(mode, 0))

    x2 = np.arange(len(modes))
    bars_sst = axes[1].bar(x2 - width/2, sst_accs,    width, label='SST (5-class)',   color='#7c3aed')
    bars_cf  = axes[1].bar(x2 + width/2, cfimdb_accs, width, label='CFIMDB (binary)', color='#0891b2')
    axes[1].set_title('SST vs CFIMDB Accuracy by Fine-tune Mode')
    axes[1].set_ylabel('Dev Accuracy')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(mode_labels)
    axes[1].set_ylim(0, 1.15)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.25)
    for bar, val in zip(list(bars_sst) + list(bars_cf), sst_accs + cfimdb_accs):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'part1_performance_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if evaluation_results:
    n = len(evaluation_results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    fig.suptitle('Confusion Matrices (Dev Split)', fontsize=13, fontweight='bold')

    for ax, (key, res) in zip(axes, evaluation_results.items()):
        matrix = np.array(res['confusion_matrix'])
        all_labels = res['all_labels']
        im = ax.imshow(matrix, cmap='Blues')
        ax.set_title(res['label'].replace(' / ', '\n'), fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_xticks(range(len(all_labels)))
        ax.set_yticks(range(len(all_labels)))
        if len(all_labels) <= 2:
            label_names = ['Neg', 'Pos']
        else:
            label_names = [str(l) for l in all_labels]
        ax.set_xticklabels(label_names, fontsize=8)
        ax.set_yticklabels(label_names, fontsize=8)
        max_val = matrix.max()
        for i in range(len(all_labels)):
            for j in range(len(all_labels)):
                color = 'white' if matrix[i, j] > max_val * 0.55 else 'black'
                ax.text(j, i, f'{matrix[i, j]:,}', ha='center', va='center',
                        color=color, fontsize=9)
        plt.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'part1_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if evaluation_results:
    # SST per-label F1
    sst_keys = [k for k in evaluation_results if 'sst' in k]
    if sst_keys:
        fig, axes = plt.subplots(1, len(sst_keys), figsize=(7 * len(sst_keys), 4))
        if len(sst_keys) == 1:
            axes = [axes]
        fig.suptitle('SST Per-Label F1 Score (Dev Split)', fontsize=13, fontweight='bold')
        sst_label_names = ['Very\nNeg', 'Neg', 'Neutral', 'Pos', 'Very\nPos']

        for ax, key in zip(axes, sst_keys):
            res = evaluation_results[key]
            f1_vals = [res['per_label_f1'].get(str(l), 0) for l in res['all_labels']]
            name_list = sst_label_names[:len(res['all_labels'])]
            bars = ax.bar(name_list, f1_vals,
                          color=['#ef4444', '#f97316', '#6b7280', '#22c55e', '#16a34a'][:len(f1_vals)])
            ax.set_title(res['label'].replace(' / ', '\n'))
            ax.set_ylabel('F1')
            ax.set_ylim(0, 1.1)
            ax.grid(axis='y', alpha=0.25)
            for bar, val in zip(bars, f1_vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / 'reports' / 'part1_sst_f1_per_label.png', dpi=150, bbox_inches='tight')
        plt.show()

    # CFIMDB per-label F1
    cf_keys = [k for k in evaluation_results if 'cfimdb' in k]
    if cf_keys:
        fig, axes = plt.subplots(1, len(cf_keys), figsize=(7 * len(cf_keys), 4))
        if len(cf_keys) == 1:
            axes = [axes]
        fig.suptitle('CFIMDB Per-Label F1 Score (Dev Split)', fontsize=13, fontweight='bold')

        for ax, key in zip(axes, cf_keys):
            res = evaluation_results[key]
            f1_vals = [res['per_label_f1'].get(str(l), 0) for l in res['all_labels']]
            cf_names = ['Negative', 'Positive']
            bars = ax.bar(cf_names[:len(f1_vals)], f1_vals, color=['#ef4444', '#22c55e'][:len(f1_vals)])
            ax.set_title(res['label'].replace(' / ', '\n'))
            ax.set_ylabel('F1')
            ax.set_ylim(0, 1.1)
            ax.grid(axis='y', alpha=0.25)
            for bar, val in zip(bars, f1_vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / 'reports' / 'part1_cfimdb_f1_per_label.png', dpi=150, bbox_inches='tight')
        plt.show()

## 12. HTML → PDF 내보내기

노트북을 실행한 뒤 이 셀을 실행하면 HTML을 거쳐 PDF로 변환합니다.

**변환 순서:**
1. `jupyter nbconvert --to html` → HTML 생성
2. `weasyprint` 또는 `pdfkit(wkhtmltopdf)` → PDF 변환

**도구 설치 (필요 시):**
```bash
pip install nbconvert weasyprint
# 또는
pip install pdfkit
brew install wkhtmltopdf  # macOS
```

In [ ]:
import subprocess
import shutil
import os

NOTEBOOK_NAME = 'PART1_sentiment_analysis'
NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / f'{NOTEBOOK_NAME}.ipynb'
HTML_OUTPUT_DIR = PROJECT_ROOT / 'reports'
HTML_PATH = HTML_OUTPUT_DIR / f'{NOTEBOOK_NAME}.html'
PDF_PATH  = HTML_OUTPUT_DIR / f'{NOTEBOOK_NAME}.pdf'

HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: ipynb → HTML (Python API 사용 — nbconvert 설치 필요)
def export_to_html():
    try:
        import nbformat
        from nbconvert import HTMLExporter
    except ImportError:
        print('[HTML] nbconvert 없음. pip install nbconvert 을 실행하세요.')
        return False

    # nbconvert 템플릿 경로를 자동 탐색
    jupyter_path_candidates = [
        '/opt/homebrew/share/jupyter',
        os.path.expanduser('~/.local/share/jupyter'),
        '/usr/local/share/jupyter',
        '/usr/share/jupyter',
    ]
    existing = [p for p in jupyter_path_candidates if Path(p).exists()]
    if existing:
        os.environ.setdefault('JUPYTER_PATH', ':'.join(existing))

    try:
        nb = nbformat.read(str(NOTEBOOK_PATH), as_version=4)
        exp = HTMLExporter()
        body, _ = exp.from_notebook_node(nb)
        with open(HTML_PATH, 'w', encoding='utf-8') as f:
            f.write(body)
        size_kb = round(HTML_PATH.stat().st_size / 1024, 1)
        print(f'[HTML] 생성 완료: {HTML_PATH} ({size_kb} KB)')
        return True
    except Exception as e:
        print(f'[HTML] 변환 실패: {e}')
        return False

html_ok = export_to_html()

In [ ]:
# Step 2: HTML → PDF
def export_to_pdf(html_path, pdf_path):
    if not Path(html_path).exists():
        print(f'[PDF] HTML 파일이 없습니다: {html_path}')
        return False

    # 방법 1: weasyprint
    try:
        from weasyprint import HTML as WeasyHTML
        WeasyHTML(filename=str(html_path)).write_pdf(str(pdf_path))
        print(f'[PDF] weasyprint 변환 완료: {pdf_path}')
        return True
    except ImportError:
        print('[PDF] weasyprint 없음, pdfkit 시도...')
    except Exception as e:
        print(f'[PDF] weasyprint 오류: {e}')

    # 방법 2: pdfkit (wkhtmltopdf 필요)
    try:
        import pdfkit
        pdfkit.from_file(str(html_path), str(pdf_path))
        print(f'[PDF] pdfkit 변환 완료: {pdf_path}')
        return True
    except ImportError:
        print('[PDF] pdfkit 없음')
    except Exception as e:
        print(f'[PDF] pdfkit 오류: {e}')

    # 방법 3: chromium/chrome headless
    for chrome_bin in ['chromium-browser', 'chromium', 'google-chrome', 'chrome']:
        chrome = shutil.which(chrome_bin)
        if chrome:
            result = subprocess.run(
                [chrome, '--headless', '--disable-gpu',
                 f'--print-to-pdf={pdf_path}',
                 str(html_path)],
                capture_output=True
            )
            if result.returncode == 0:
                print(f'[PDF] Chrome headless 변환 완료: {pdf_path}')
                return True

    print('[PDF] PDF 변환 도구를 찾지 못했습니다.')
    print('      아래 중 하나를 설치하세요:')
    print('      pip install weasyprint')
    print('      pip install pdfkit && brew install wkhtmltopdf')
    print(f'      또는 브라우저에서 {html_path} 를 열고 인쇄 → PDF로 저장하세요.')
    return False

if html_ok:
    pdf_ok = export_to_pdf(HTML_PATH, PDF_PATH)

    if pdf_ok:
        size_mb = round(PDF_PATH.stat().st_size / (1024 * 1024), 2)
        print(f'\n최종 산출물:')
        print(f'  HTML: {HTML_PATH}')
        print(f'  PDF:  {PDF_PATH} ({size_mb} MB)')
else:
    print('\nHTML 변환이 필요합니다. 위 셀에서 jupyter를 설치 후 재실행하세요.')

## 13. 실행 메모

### GPT-2 아키텍처 핵심 파일
| 파일 | 역할 |
|------|------|
| `modules/attention.py` | Causal Self-Attention (Scaled Dot-Product + Causal Mask) |
| `modules/gpt2_layer.py` | Pre-LayerNorm Transformer Block (Attention + FFN + Residual) |
| `models/gpt2.py` | 전체 GPT-2 모델 (Embedding + N×Layer + LM Head) |
| `models/base_gpt.py` | HuggingFace 가중치 로드 베이스 클래스 |

### 감정 분석 파이프라인
- **GPT-2 마지막 토큰 임베딩** → Dropout → Linear → 감정 클래스 로짓
- `last-linear-layer`: GPT-2 파라미터 동결, 분류 레이어만 학습 (lr=1e-3)
- `full-model`: GPT-2 전체 fine-tuning (lr=1e-5)

### 기준 성능 (Baseline)
- SST last-linear-layer dev acc ≈ **0.462**
- SST full-model dev acc ≈ **0.513**
- CFIMDB last-linear-layer dev acc ≈ **0.861**
- CFIMDB full-model dev acc ≈ **0.976**

### 제출 파일 목록
```
predictions/last-linear-layer-sst-dev-out.csv
predictions/last-linear-layer-sst-test-out.csv
predictions/full-model-sst-dev-out.csv
predictions/full-model-sst-test-out.csv
predictions/last-linear-layer-cfimdb-dev-out.csv
predictions/last-linear-layer-cfimdb-test-out.csv
predictions/full-model-cfimdb-dev-out.csv
predictions/full-model-cfimdb-test-out.csv
```